# Save a European map of BMR on SHERPA resolution

In [1]:
import os
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
EU_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [3]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [4]:
# === Set GBD version ===
GBD_version = "GBD23"

In [5]:
# Load one SHERPA file for lat/lon reference (still needed for output coords/shape check)
eu_file = "EU_concentration_H_2040.nc"
eu_path = os.path.join(EU_DIR, eu_file)
eu = xr.open_dataarray(eu_path)

# Load pre-computed EU-extent, SHERPA-resolution country mask: (country, lat, lon)
eu_mask_file = "GBD_Country_Masks_EU_SHERPA_res.nc"
eu_mask_path = os.path.join(MASKS_DIR, eu_mask_file)
eu_mask = xr.open_dataarray(eu_mask_path)

# Sanity check: confirm the mask actually matches the EU grid you're about to write output on.
assert eu_mask.sizes["latitude"] == eu.sizes["latitude"], "lat size mismatch vs EU target grid!"
assert eu_mask.sizes["longitude"] == eu.sizes["longitude"], "lon size mismatch vs EU target grid!"
assert np.allclose(eu_mask.latitude.values, eu.latitude.values, atol=1e-4), "lat values don't align!"
assert np.allclose(eu_mask.longitude.values, eu.longitude.values, atol=1e-4), "lon values don't align!"

print(f"Loaded eu_mask: {dict(eu_mask.sizes)}")

Loaded eu_mask: {'latitude': 781, 'longitude': 601, 'country': 204}


In [17]:
for health_VAR in health_vars:
    # Load BMR for health_var
    bmr_file = f"{GBD_version}_BMR_Country_{health_VAR}_newlabels_2015-2019.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    bmr = xr.open_dataarray(bmr_path)

    bmr_country_mask = (eu_mask * bmr).sum("country")

    out_file = f"{GBD_version}_BMR_European_Country_Map_{health_VAR}_2015-2019.nc"
    out_path = os.path.join(BMR_DIR, out_file)
    print(f"Saving to {out_path}")
    bmr_country_mask.to_netcdf(out_path)

print("All processing complete.")

Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_COPD_2015-2019.nc
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_DIABETES_2015-2019.nc
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_ISCHEMIC_HEART_DISEASE_2015-2019.nc
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_LOWER_RESPIRATORY_INFECTIONS_2015-2019.nc
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_LUNG_CANCER_2015-2019.nc
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_STROKE_2015-2019.nc
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_DEMENTIA_2015-2019.nc
All processing complete.
